In [4]:
import os
import glob
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [5]:
import sys
print(sys.executable)

/home/AC/pyspark-venv/bin/python


In [ ]:
# Q1 - Install Spark and PySpark
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("jupyter-pyspark")
    .getOrCreate()
)

spark.version

'4.1.1'

In [7]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -O yellow_tripdata_2025-11.parquet

In [8]:
!wget -q https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -O taxi_zone_lookup.csv

In [9]:
!ls -lh yellow_tripdata_2025-11.parquet taxi_zone_lookup.csv

-rw-r--r-- 1 AC AC 13K Feb 22  2024 taxi_zone_lookup.csv
-rw-r--r-- 1 AC AC 68M Dec 19 15:51 yellow_tripdata_2025-11.parquet


In [10]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.printSchema()
df.count()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



4181444

In [11]:
# Q2 - Read the November 2025 Yellow into a Spark Dataframe. Repartition the Dataframe to 4 partitions and save it to parquet.
import os, glob, shutil

out_dir = "yellow_2025_11_4part"
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)

df.repartition(4).write.mode("overwrite").parquet(out_dir)

files = glob.glob(os.path.join(out_dir, "*.parquet"))
sizes_mb = [os.path.getsize(f) / (1024*1024) for f in files]
avg_mb = sum(sizes_mb) / len(sizes_mb)

len(files), avg_mb

(4, 25.331268072128296)

In [14]:
# Q3 - How many taxi trips were there on the 15th of November?
df_Nov_15 = df.filter(F.to_date("tpep_pickup_datetime") == F.lit("2025-11-15"))
df_Nov_15.count()

162604

In [17]:
# Q4 - What is the length of the longest trip in the dataset in hours?
df_dur = df.withColumn(
    "trip_hours",
    (F.col("tpep_dropoff_datetime").cast("timestamp").cast("long")
     - F.col("tpep_pickup_datetime").cast("timestamp").cast("long")) / 3600.0
)
df_dur.agg(F.max("trip_hours").alias("max_hours")).show()

[Stage 13:>                                                         (0 + 2) / 2]

+-----------------+
|        max_hours|
+-----------------+
|90.64666666666666|
+-----------------+



In [18]:
# Q5 - Spark's User Interface which shows the application's dashboard runs on which local port?
spark.sparkContext.uiWebUrl

'http://spark-hw-ac.us-central1-a.c.dtc-de-course-486019.internal:4041'

In [19]:
# Q6 - Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?
zones = (spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("taxi_zone_lookup.csv"))

joined = df.join(zones, df.PULocationID == zones.LocationID, "inner")

least = (joined.groupBy("Zone")
         .count()
         .orderBy(F.col("count").asc(), F.col("Zone").asc())
         .limit(10))

least.show(truncate=False)

[Stage 19:>                                                         (0 + 2) / 2]

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Arden Heights                                |1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Governor's Island/Ellis Island/Liberty Island|1    |
|Port Richmond                                |3    |
|Great Kills                                  |4    |
|Green-Wood Cemetery                          |4    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
+---------------------------------------------+-----+



In [20]:
spark.stop()